# LLM graphs with loops

## Introduction

This document has simple examples of LLM-graphs with cycles (or loops).

---

## Setup

In [1]:
use LLM::Functions;
use LLM::Graph;

use Markdown::Grammar;

In [2]:
my $conf = llm-configuration('chatgpt', model => 'gpt-5.6-luna', :8192max-tokens);

LLM::Configuration(:name("chatgpt"), :model("gpt-5.6-luna"), :module("WWW::OpenAI"), :max-tokens(8192))

In [3]:
%*ENV.sort(*.key).grep(*.key.contains('_KEY')).elems

8

---

## Simplest, one-node loop (deteministic)

- Repetition is the simplest loop - it's achieved by self-reference.

- Once node state initialized, such a loop can be conceptually unrolled into a bounded sequence of passes.

```mermaid
flowchart LR
    B0 --> B1 --> B2 --> Next["..."]
```

- Initialization with "initial-value" element of the graph node spec.

- The LLM-graphs can be seen "just" as "executable graphs" (without LLM involvement)


Greate a deteministic graph with one node that has a loop:

In [4]:
my %rules =
        refine => {
            eval-function => sub ($refine) { $refine + 1 },
            initial-value => sub ($seed) { $seed },
            max-iterations => 3
        };

my $g = LLM::Graph.new(%rules, :!async);

LLM::Graph(size => 1, nodes => refine)

Plot the graph:

In [5]:
#% html
$g.create-graph.dot(:svg, theme => 'ortho', vertex-width => 0.6);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- refine -->
 
 refine 
 
 refine 
 
<!-- refine->refine -->
 
 refine->refine

Evaluate the graph and show evaluation info:

In [6]:
$g.eval(seed => 10).nodes<refine>;
.say for |$g.nodes<refine>.grep({ $_.value ~~ Numeric:D | Str:D }).sort

iteration-count => 3
max-iterations => 3
result => 13
termination-reason => max-iterations


Evaluate the graph again -- the value given to "refine" overrides the value given to to "seed":

In [7]:
say $g.eval(seed => 10, refine => 20).nodes<refine><result>;

23


All info for the node "refine":

In [8]:
.say for |$g.nodes<refine>.sort

eval-function => sub { }
initial-value => sub { }
initial-value-input => [seed]
input => [refine]
iteration-count => 3
max-iterations => 3
result => 23
spec-type => (Callable)
termination-function-input => []
termination-reason => max-iterations
test-function-input => []


---

## Simplest, one-node loop with max-iterations (LLM)

In [9]:
my $refined = llm-graph({
    draft => sub ($topic, $field) {
        "Write a few sentence explanation of $topic for first-year undergraduate $field students." 
    },
    refine => {           
        eval-function => sub ($refine) { 
            llm-synthesize(
                "Improve the following explanation to be clearer, more precise, and more concise:\n\n$refine",
                e => $conf
            )
        },
        initial-value => sub ($draft) { $draft },
        max-iterations => 3
    }
},
e => $conf
);

LLM::Graph(size => 2, nodes => draft, refine)

In [10]:
#% html
$refined.create-graph;
$refined.dot(:svg, theme => 'ortho', vertex-width => 1.2)

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- draft -->
 
 draft 
 
 
 draft 
 
<!-- refine -->
 
 refine 
 
 refine 
 
<!-- draft->refine -->
 
 draft->refine 
 
 
 4 
 
<!-- refine->refine -->
 
 refine->refine

In [11]:
$refined.eval({ topic => "quantile regression", field => "mathematical fitting of bassis functions"})

LLM::Graph(size => 2, nodes => draft, refine)

Display the result:

In [12]:
#% markdown
$refined.nodes<refine><result>
.subst('\(', '$', :g)
.subst('\)', '$', :g)
.subst('\[', '$$', :g)
.subst('\]', '$$', :g)

Quantile regression estimates a specified **conditional quantile** of the response, rather than its conditional mean. For a predictor value $x$, median regression estimates the conditional 50th percentile of $Y$. More generally, the $q$-th conditional quantile $Q_Y(q\mid x)$ satisfies
$$
P\!\left(Y\le Q_Y(q\mid x)\mid X=x\right)\approx q.
$$
Thus, a 90th-percentile regression estimates the value below which approximately 90% of responses fall, conditional on $X=x$.

Using basis functions, the quantile is modeled as
$$
\widehat Q_Y(q\mid x)=\sum_j \beta_j(q)\,\phi_j(x),
$$
where $\phi_j(x)$ are known basis functions and the coefficients $\beta_j(q)$ are estimated by minimizing the pinball loss
$$
\sum_i \rho_q\!\left(y_i-\widehat Q_Y(q\mid x_i)\right),
\qquad
\rho_q(u)=u\bigl(q-\mathbf 1\{u<0\}\bigr).
$$

Unlike mean regression, quantile regression describes how different parts of the conditional distribution change with $x$. It is therefore useful for capturing heteroscedasticity and estimating conditional quantile bands, including prediction intervals when appropriate.

Display node evaluation info:

In [13]:
.say for |$refined.nodes<refine>.grep(*.key ∈ <iteration-count termination-reason>)

iteration-count => 3
termination-reason => max-iterations


---

## One-node loop with max-iterations and termination function

In [14]:
use JSON::Fast;

my $retry = llm-graph({
    json-record => {           
        eval-function => sub ($json-record) {
            # say "eval-function: ", (:$json-record);
            llm-synthesize(
                "Extract the person's name and birth year from this text as a JSON object with keys \"name\" and \"year\". Reply with the JSON only.\n\n$json-record",
                e => $conf
            )
        },
        initial-value => sub ($text) { $text },
        termination-function => sub ($json-record) {
            # say "termination-function: ", (:$json-record);
            my $h = Nil;
            try { $h = from-json($json-record) } 
            $! ?? False !! $h ~~ Map:D #&& $h.keys.sort eq <name year>
        },
        max-iterations => 3
    },
    parsed => { eval-function => sub ($json-record) { from-json($json-record) } }
},
e => $conf
);

LLM::Graph(size => 2, nodes => json-record, parsed)

In [15]:
#% html
$retry.create-graph;
$retry.dot(:svg, theme => 'ortho', vertex-width => 1.2)

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- json-record -->
 
 json-record 
 
 json-record 
 
<!-- json-record->json-record -->
 
 json-record->json-record 
 
 
 
<!-- parsed -->
 
 parsed 
 
 parsed 
 
<!-- json-record->parsed -->
 
 json-record->parsed

In [21]:
$retry.eval({text => 'Dimitar Savatinov is born in 1980 in Bulgaria.'}).nodes<parsed><result>

{name => Dimitar Savatinov, year => 1980}

Here is the info for the evaluation of loop node:

In [22]:
.say for |$retry.nodes<json-record>.sort

eval-function => sub { }
initial-value => sub { }
initial-value-input => [text]
input => [json-record]
iteration-count => 1
max-iterations => 3
result => {"name":"Dimitar Savatinov","year":1980}
spec-type => (Callable)
termination-function => sub { }
termination-function-input => [json-record]
termination-reason => termination-function
test-function-input => []


---

## Actor-Critic loop

- With multiple nodes loop, a starting point is needed.

- It can be "unrolled" once properly initialized:

```mermaid
flowchart LR
    B0 --> C1 --> A2 --> B1 --> Next["..."]
```

- A loop is anchored when one node in the cycle provides the initial state and therefore determines where the first pass begins.

- The starting point (i.e. the loop anchor) is designated implicitly by providing initialization or other loop controls in its specifications.


In [23]:
my $actorCritic = llm-graph({
	draft => sub ($idea) { "Write a first draft of a short product description for $idea ."},
    critique => sub ($refine) { 
        "Review this draft and list concrete problems:\n\n$refine\n\n" ~ 
        "If it needs no further work, reply with exactly APPROVED."
    },
    refine => {
      	eval-function => sub ($refine, $critique) {
			say (:$refine);
			say (:$critique);
			say '-' x 50;
			llm-synthesize("Improve this draft:\n\n$refine\n\nusing this critique:\n\n\$critique", e => $conf) 
		},
      	initial-value => sub ($draft) { $draft },
		termination-function => sub ($critique) { $critique ~~ Str:D && $critique.contains('APPROVED') },
      	max-iterations => 4
    }
});

LLM::Graph(size => 3, nodes => critique, draft, refine)

In [24]:
#% html
$actorCritic.create-graph;
$actorCritic.dot(:svg, theme => 'ortho', vertex-width => 1.2)

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- critique -->
 
 critique 
 
 
 critique 
 
<!-- refine -->
 
 refine 
 
 refine 
 
<!-- critique->refine -->
 
 critique->refine 
 
 
 9 
 
<!-- draft -->
 
 draft 
 
 
 draft 
 
<!-- draft->refine -->
 
 draft->refine 
 
 
 4 
 
<!-- refine->critique -->
 
 refine->critique 
 
 
 
<!-- refine->refine -->
 
 refine->refine

In [ ]:
$actorCritic.eval({idea => 'Wolfram Language LLM prompt that knows the schedule of a conference.'})

Display the result:

In [26]:
#% markdown
$actorCritic.nodes<refine><result>

**Plan Your Perfect Conference Day—Just Ask**

Explore the entire conference schedule in natural language with a Wolfram Language LLM prompt. Ask about sessions, speakers, times, and locations—or discover events based on your interests. Then quickly build a personalized agenda that fits your goals.

Spend less time searching through the schedule and more time connecting, learning, and getting the most out of your conference.

In [27]:
.say for |$actorCritic.nodes<refine>.grep(*.key ∈ <max-iterations termination-reason>).sort

max-iterations => 4
termination-reason => max-iterations
